> **中文阅读版**：本文件是 `02_tool_contracts.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# 工具契约与来源（Provenance）

**Docker 镜像**：`ml4t`

智能体的能力上限取决于其工具返回的内容。在本章构建的预测系统中，搜索是唯一工具，因此其契约决定了哪些证据能够到达模型、这些证据在被预测日当天是否本来可被读取，以及最终推理（rationale）中的某项主张能否追溯到某份文档。本 notebook 写定该契约及其上层控制。

**学习目标**：
- 在协议之后定义搜索工具，使提供方可替换而无需改动智能体
- 按截止日期过滤结果，使针对已解决问题评分的智能体无法读到答案，并将过滤器无法裁定的无日期结果分离出来
- 限制智能体可读取的发布者，并在执行日志中记录该策略移除了多少证据
- 将一份工具定义渲染为 Anthropic 和 OpenAI 的 schema 格式
- 读取执行日志，查看一次运行实际调用了什么，而非模型声称自己在做什么

**本书参考**：第 24 章，第 24.4 节（工具集成：契约、控制与上下文工程）

**前置条件**：[`01_react_reasoning`](01_react_reasoning.ipynb)（LLM 提供方与 ReAct 模式）。

In [2]:
"""Tool Contracts and Provenance - search protocol, audit trails, and domain policy."""

import sys

from utils.paths import get_chapter_dir

sys.path.insert(0, str(get_chapter_dir(24)))

import json
from datetime import date
from urllib.parse import urlparse

import polars as pl
from agent_fixtures import get_demo_question
from agent_schemas import SearchResult
from agent_tools import (
    DEFAULT_ALLOWED_DOMAINS,
    SEARCH_TOOL,
    ToolExecutor,
    create_search_client,
    format_search_results,
)

In [3]:
SEARCH_PROVIDER = "mock"  # explicitly synthetic contract fixture; "tavily" for live retrieval
MAX_RESULTS = 5

## SearchClient 协议

本章构建的智能体只有一个工具：网络搜索。所有能够提供该工具的提供方，无论是 Tavily、确定性的 mock，还是后续新增的实现，都通过同一个方法签名访问：

```python
class SearchClient(Protocol):
    def search(
        self, query: str, max_results: int = 5, cutoff_date: date | None = None
    ) -> list[SearchResult]: ...
```

该签名中有两点比抽象本身更有价值。它返回类型化的 `SearchResult` 对象，而不是提供方的原始 JSON，因此下游不依赖供应商的字段名。而 `cutoff_date` 被纳入契约，而非事后附加，这正是让 Point-in-Time (PIT) 过滤成为工具职责而非每个调用方职责的原因。

### 一条结果包含什么

一个 `SearchResult` 记录证据来自何处以及它说了什么。**Provenance（溯源）** 就是该来源记录：URL 标识发布者，并使某项主张可追溯到其来源页面；`published` 则是下文过滤所依赖的日期。`score` 是提供方自身的相关性排序，只用于排列结果，并不说明结果是否真实。

In [4]:
example = SearchResult(
    title="Fed officials signal patience on rate cuts",
    url="https://federalreserve.gov/press-releases",
    snippet="Multiple FOMC members indicate no urgency to cut rates.",
    published="2025-03-05",
    score=0.96,
)
print(f"Title:     {example.title}")
print(f"URL:       {example.url}")
print(f"Published: {example.published}")
print(f"Score:     {example.score}")

Title:     Fed officials signal patience on rate cuts
URL:       https://federalreserve.gov/press-releases
Published: 2025-03-05
Score:     0.96


## 运行搜索

`SEARCH_PROVIDER` 决定结果来自何处，且需显式命名而非推断。若保留为 `mock`，它会构建 `MockSearchClient`，其结果来自 `agent_tools.py` 中的固定字典，因此本 notebook 在每台机器上产生相同输出且不产生费用。若指定真实提供方（本章为 `tavily`），则需要该提供方的密钥；密钥缺失会抛出异常，而不是悄悄回退到固定测试数据（fixture），从而避免将合成结果误读为检索结果。

演示问题询问 NVIDIA 是否超出了其 2025 财年第四季度盈利预期，公司于 2025-02-26 发布财报时给出了答案。该问题带有 **截止日期（cutoff date）** 2025-02-20：这是搜索工具强制执行的边界，会丢弃在该日或之后发布的任何内容，因此回答该问题的智能体只能读取该日之前已存在的信息。

In [5]:
search = create_search_client(SEARCH_PROVIDER)
print("Mode: SYNTHETIC contract fixture" if SEARCH_PROVIDER == "mock" else "Mode: LIVE retrieval")
print(f"Provider: {type(search).__name__}\n")

question = get_demo_question()
cutoff = date.fromisoformat(question.cutoff_date)
print(f"Question: {question.question}")
print(f"Cutoff:   {cutoff}\n")

results = search.search(
    "NVIDIA Q4 earnings expectations", max_results=MAX_RESULTS, cutoff_date=cutoff
)
print(f"Results: {len(results)}")
for r in results:
    print(f"  [{r.published or '?'}] {r.title}")
    if r.snippet:
        print(f"    {r.snippet[:100]}")

Mode: SYNTHETIC contract fixture
Provider: MockSearchClient

Question: Will NVIDIA beat Q4 FY2025 earnings expectations?
Cutoff:   2025-02-20

Results: 2
  [2025-02-17] Synthetic fixture: NVIDIA suppliers signal continued AI server demand
    Supply-chain checks point to resilient GPU demand ahead of earnings report.
  [2025-02-18] Synthetic fixture: Analysts raise NVIDIA targets ahead of earnings
    Street revisions remain constructive ahead of the Feb 26 report.


## Point-in-Time (PIT) 过滤

上述问题询问的是一个已经解决的事件，而这是预测智能体唯一可被评分的问题类型。这也是那种极易因偶然因素答对的类型：今天在开放网络上搜索“NVIDIA 是否超出 2025 财年第四季度盈利预期”，第一个结果就会给出答案。以此方式评估的智能体衡量的是搜索索引，而不是预测能力。

`cutoff_date` 就是防线。客户端会丢弃发布日期在截止日期当天或之后的任何结果，因此到达智能体的内容日期严格早于该日。问题自身的 `cutoff_date` 为 2025-02-20，而 NVIDIA 于 2025-02-26 发布财报，因此智能体基于截至并包括 2025-02-19 已发布的信息工作，而答案在一周后才出现。

过滤器仅保留带有可解析发布日期且早于截止日期的结果，因此无日期条目会被丢弃而不是被信任。它无法做的是确认页面在其声称日期当天的内容，也无法移除模型已从训练中获得的知识。截止日期使检索具备 Point-in-Time (PIT) 属性；但不会使预测本身也成为 Point-in-Time (PIT)。

In [6]:
# Search WITHOUT cutoff - all results returned
all_results = search.search("NVIDIA Q4 earnings", max_results=MAX_RESULTS)
print(f"Without cutoff: {len(all_results)} results")
for r in all_results:
    print(f"  [{r.published or '?'}] {r.title}")

print()

# Search WITH cutoff - future results filtered out
filtered_results = search.search("NVIDIA Q4 earnings", max_results=MAX_RESULTS, cutoff_date=cutoff)
print(f"With cutoff ({cutoff}): {len(filtered_results)} results")
for r in filtered_results:
    print(f"  [{r.published or '?'}] {r.title}")

Without cutoff: 3 results
  [2025-02-17] Synthetic fixture: NVIDIA suppliers signal continued AI server demand
  [2025-02-18] Synthetic fixture: Analysts raise NVIDIA targets ahead of earnings
  [2025-02-20] Synthetic fixture: Investors watch valuation risk as AI rally broadens

With cutoff (2025-02-20): 2 results
  [2025-02-17] Synthetic fixture: NVIDIA suppliers signal continued AI server demand
  [2025-02-18] Synthetic fixture: Analysts raise NVIDIA targets ahead of earnings


## 为智能体格式化结果

结果以工具消息中的文本形式到达模型，因此格式化也是契约的一部分。`format_search_results` 为每个结果编号，并将标题、URL、摘要片段和发布日期分别置于单独的行。URL 和日期之所以存在，是为了让模型能够权衡一手来源与博客、近期报道与过时报道，并且使最终推理（rationale）中的引文可追溯到其来源文档。

**发现**：合成过滤器会排除其日期边界之外的记录。历史模拟还需要在预测截止时点可获得的证据，以及对模型先验知识的评估。这些 fixture 结果检验的是过滤器，而不是历史预测能力。

In [7]:
formatted = format_search_results(filtered_results)
print(formatted[:500])

SEARCH RESULTS:
1. Synthetic fixture: NVIDIA suppliers signal continued AI server demand
   URL: https://example.org/nvidia-0
   Supply-chain checks point to resilient GPU demand ahead of earnings report.
   Published: 2025-02-17

2. Synthetic fixture: Analysts raise NVIDIA targets ahead of earnings
   URL: https://example.org/nvidia-1
   Street revisions remain constructive ahead of the Feb 26 report.
   Published: 2025-02-18



## 执行审计轨迹

`ToolExecutor` 封装搜索客户端并记录每次调用：查询、是否成功、耗时以及由哪个客户端提供服务。该记录与 agent 的推理轨迹有意分开保存。推理轨迹是模型声称自己正在做什么；执行日志则是实际发生了什么。调试糟糕预测通常从二者之间的差异开始。

In [8]:
executor = ToolExecutor(search=search)

# Execute several searches
r1 = executor.execute_search("NVIDIA earnings Q4 2025", max_results=3, cutoff_date=cutoff)
r2 = executor.execute_search("semiconductor demand AI servers", max_results=3, cutoff_date=cutoff)
r3 = executor.execute_search("NVIDIA valuation risk", max_results=3, cutoff_date=cutoff)

print(f"Searches executed: {len(executor.execution_log)}\n")

pl.DataFrame(
    {
        "query": [entry.args.get("query", "?")[:38] for entry in executor.execution_log],
        "status": [entry.status for entry in executor.execution_log],
        "duration_ms": [round(entry.duration_ms, 1) for entry in executor.execution_log],
        "result_preview": [entry.result_preview for entry in executor.execution_log],
    }
)

Searches executed: 3



query,status,duration_ms,result_preview
str,str,f64,str
"""NVIDIA earnings Q4 2025""","""success""",0.2,"""2 results"""
"""semiconductor demand AI server…","""success""",0.0,"""2 results"""
"""NVIDIA valuation risk""","""success""",0.0,"""2 results"""


耗时四舍五入为零，是因为 mock 客户端从内存中的字典返回。面对实时提供商时，同一列会承载网络往返时间；agent 的大部分 wall-clock 时间都花在这里，也是必须设置单次调用超时的地方。

## 来源策略

agent 可以阅读哪些发布者是一个研究决策，而不是提示词细节。允许列表（allowlist）列出其报道可作为证据的域名；阻止列表（blocklist）列出不能作为证据的域名。在 `agent_tools.py` 中，`apply_domain_policy` 实现了二者：在去掉开头的 `www.` 后，按注册域名及其子域名进行匹配，因此 `reuters.com` 也会覆盖 `uk.reuters.com`。先应用阻止列表，再应用允许列表。

`DEFAULT_ALLOWED_DOMAINS` 是本章的起始集合：通讯社、金融报纸，以及其他机构报道其数字时所依据的一手来源。

**观察**：这些合成调用展示的是审计记录，而不是提供商延迟。要从真实实时抓取中检查测得的耗时来识别慢查询；fixture 的计时无法预测实时服务。

## 域名策略

工具执行器在将返回的 URL 提供给 agent 之前，会按主机名进行过滤。这是一种访问约束，并不证明保留下来的来源为真。下一个本地示例是一个带标签的合成域名过滤 fixture。

In [9]:
assert len(DEFAULT_ALLOWED_DOMAINS) == 11, "DEFAULT_ALLOWED_DOMAINS count drifted"
pl.DataFrame({"allowed_domain": sorted(DEFAULT_ALLOWED_DOMAINS)})

allowed_domain
str
"""bloomberg.com"""
"""bls.gov"""
"""cmegroup.com"""
"""cnbc.com"""
"""coindesk.com"""
…
"""ft.com"""
"""nasdaq.com"""
"""reuters.com"""


### 自定义域名策略——可观察的执行

此处的策略执行是**检索后（post-retrieval）**的：搜索提供商返回任何与查询匹配的内容，agent 的工具层在把结果交回之前，会过滤掉注册域名不在允许列表上（或出现在阻止列表上）的结果。下面的 fixture 为一个 mock 搜索预置结果，其中同时包含一个 `reddit.com` URL 和一个 `wsj.com` URL；实例化两个策略，然后观察被阻止的那个消失。

In [10]:
mixed_results = [
    SearchResult(
        title="Synthetic Fed reaction on rates",
        url="https://www.reuters.com/markets/fed-reaction",
        snippet="...",
        published="2025-03-04",
        score=0.91,
    ),
    SearchResult(
        title="Synthetic WSJ analysis: NVIDIA earnings setup",
        url="https://www.wsj.com/articles/nvidia",
        snippet="...",
        published="2025-03-05",
        score=0.88,
    ),
    SearchResult(
        title="Synthetic r/wallstreetbets: NVDA hot take",
        url="https://www.reddit.com/r/wallstreetbets/x",
        snippet="...",
        published="2025-03-05",
        score=0.40,
    ),
]


def apply_domain_policy(
    items: list[SearchResult],
    *,
    allowed: set[str] | None = None,
    blocked: set[str] | None = None,
) -> list[SearchResult]:
    """Drop results whose host is not on `allowed`, or is on `blocked`."""
    keep: list[SearchResult] = []
    for r in items:
        host = urlparse(r.url).netloc.removeprefix("www.")
        if blocked and host in blocked:
            continue
        if allowed and host not in allowed:
            continue
        keep.append(r)
    return keep


financial_only = apply_domain_policy(
    mixed_results, allowed={"reuters.com", "wsj.com", "ft.com", "bloomberg.com"}
)
no_social = apply_domain_policy(mixed_results, blocked={"reddit.com", "twitter.com"})

pl.DataFrame(
    {
        "policy": ["raw retrieval", "allowlist (financial)", "blocklist (social)"],
        "n_results": [len(mixed_results), len(financial_only), len(no_social)],
        "hosts_kept": [
            ", ".join(urlparse(r.url).netloc for r in mixed_results),
            ", ".join(urlparse(r.url).netloc for r in financial_only),
            ", ".join(urlparse(r.url).netloc for r in no_social),
        ],
    }
)

policy,n_results,hosts_kept
str,i64,str
"""raw retrieval""",3,"""www.reuters.com, www.wsj.com, …"
"""allowlist (financial)""",2,"""www.reuters.com, www.wsj.com"""
"""blocklist (social)""",2,"""www.reuters.com, www.wsj.com"""


## 工具 Schema 转换

`ToolDefinition` 类将搜索工具 schema 渲染为 Anthropic 和 OpenAI 两种格式。LLM 正是通过这种方式知道有哪些工具可用以及如何调用它们。相同的定义，不同的提供商格式。

In [11]:
print(f"Tool: {SEARCH_TOOL.name}")
print(f"Description: {SEARCH_TOOL.description}")
print(f"Parameters: {json.dumps(SEARCH_TOOL.parameters, indent=2)}")
print(f"Required: {SEARCH_TOOL.required}")

Tool: search
Description: Search the web for recent news, analysis, and data relevant to a forecasting question.
Parameters: {
  "query": {
    "type": "string",
    "description": "Search query string"
  }
}
Required: ['query']


### Anthropic 格式

Anthropic 在顶层使用 `input_schema`。

In [12]:
print(json.dumps(SEARCH_TOOL.to_anthropic_schema(), indent=2))

{
  "name": "search",
  "description": "Search the web for recent news, analysis, and data relevant to a forecasting question.",
  "input_schema": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "Search query string"
      }
    },
    "required": [
      "query"
    ]
  }
}


### OpenAI 格式

OpenAI 将 schema 包装在一个带有 `type: "function"` 外层结构的 `function` 对象中。

In [13]:
print(json.dumps(SEARCH_TOOL.to_openai_schema(), indent=2))

{
  "type": "function",
  "function": {
    "name": "search",
    "description": "Search the web for recent news, analysis, and data relevant to a forecasting question.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "Search query string"
        }
      },
      "required": [
        "query"
      ]
    }
  }
}


**观察**：结构差异很小（Anthropic 使用 `input_schema`，OpenAI 包装在 `function` 对象中）。与提供商无关的工具定义可防止提供商特定的 schema 不匹配。

## 工具不可用时

部署在没有出站访问权限的网络中的 agent，或在提供商宕机期间运行的 agent，仍然必须做点什么。`ToolExecutor` 会将缺失的客户端视为一个被记录的结果，而不是异常：该调用以状态 `disabled` 记录下来并返回空结果，因此 agent 的循环继续运行，记录也说明了证据为何缺失。这里的异常会中止运行，并留下无可审计的内容。

In [14]:
disabled_executor = ToolExecutor(search=None)
empty = disabled_executor.execute_search("test query")
print(f"Results when disabled: {len(empty)}")
print(f"Log entry: {disabled_executor.execution_log[0].status}")

Results when disabled: 0
Log entry: disabled


## 关键要点

1. **工具契约决定了智能体能够知道什么。** 所有下游内容（包括预测）都是搜索返回内容的函数，因此该契约值得像提示词一样投入同等的设计关注。
2. **发布日期过滤器是回测与演示之间的分界线。** 没有它，在已解决问题上评估的智能体会读到答案。
3. **没有日期的结果不是截止日期之前的结果。** 在记录中将两者分开，并明确决定策略；默默将无日期证据计为 Point-in-Time (PIT) 安全，正是无后见之明声明出错的方式。
4. **在工具层而非提示词中执行来源策略。** 被要求不要阅读社交媒体的模型有时仍会阅读；而一个永远不会到达模型的宿主则无法做到。
5. **执行日志是实际运行内容的记录**，独立于模型对其正在做什么的叙述。当两者不一致时，应信任日志。
6. **一份工具定义，按提供商分别渲染。** Anthropic 与 OpenAI 之间的 schema 差异是机械性的，而单一的 `ToolDefinition` 可防止它们变成两个逐渐漂移的事实来源。

**此处构建内容的已知局限。** 来源策略在检索之后运行，因此被阻止的文档仍会被抓取并产生费用；在 API 提供相应功能时，提供商侧的域名过滤器更便宜。允许列表是一份发布者列表，这是对证据质量的粗略代理：允许域名上的通讯社联合报道与被阻止域名上的同一报道是相同证据。这里没有对重复同一底层来源的结果去重，因此智能体可能把一个报道的五份副本误认为五次独立确认。

**下一步**：[`03_state_and_memory`](03_state_and_memory.ipynb) 将智能体的状态显式化，使一次运行可以被检查点、重放，并根据证据质量进行门控。

**本书**：第 24.4 节涵盖工具契约设计、Model Context Protocol 和沙箱模式。